# 📓 Notebook 08 — ReAct Agent (Multi-Step Reasoning)
---
**Phase 3 · Tools & Function Calling**

> ReAct = **Reasoning + Acting**. The agent thinks, acts, observes, and repeats until it has the answer.

### What You'll Learn
| Topic | Why It Matters |
|-------|---------------|
| ReAct pattern | Thought → Action → Observation loop |
| Multi-step reasoning | Chain tool calls to solve complex problems |
| Trace logging | Track the agent's reasoning process |
| Agent architectures | ReAct vs CoT vs Plan-then-Execute |

### 🏗️ Build: Research Assistant Agent

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")

---
## 2. The ReAct Pattern

### Thought → Action → Observation

```
User: "What's the population of India divided by the area in square km?"

Thought 1: I need to search for India's population and area.
Action 1:  search("India population 2024")
Observation 1: India's population is approximately 1.44 billion.

Thought 2: Now I need the area.
Action 2:  search("India area square kilometers")
Observation 2: India's area is 3,287,263 sq km.

Thought 3: Now I can calculate: 1,440,000,000 / 3,287,263
Action 3:  calculate("1440000000 / 3287263")
Observation 3: Result: 438.0

Final: India's population density is approximately 438 people per square kilometer.
```

> **Interview Critical:** ReAct outperforms pure CoT because it can **ground reasoning in real tool outputs** rather than hallucinating intermediate facts.

### 🖼️ The ReAct Loop — Illustrated

![The ReAct Loop: Think → Act → Observe](images/react_loop_diagram.png)

#### 💡 Intuition: ReAct = Google Search + Thinking Out Loud

Imagine you're researching a topic:
1. 🧠 **Think:** "I need to find India's population."
2. ⚡ **Act:** You Google it → `search("India population")`
3. 👁️ **Observe:** You read the result → "1.44 billion"
4. 🧠 **Think:** "Now I need the area to calculate density."
5. ⚡ **Act:** You Google again → `search("India area km²")`
6. 👁️ **Observe:** "3,287,263 km²"
7. 🧠 **Think:** "I can now calculate: 1.44B / 3.29M = ~438 people/km²"
8. ✅ **Done!** → Final answer

ReAct agents do exactly this — but with LLMs doing the thinking and tools doing the acting.

In [ ]:
from IPython.display import HTMLHTML("""<div id="react-demo" style="font-family:'Segoe UI',system-ui,sans-serif;background:linear-gradient(135deg,#0f0c29,#1a1a2e);padding:24px;border-radius:16px;color:#e0e0e0;max-width:660px;box-shadow:0 8px 32px rgba(0,0,0,0.4);">  <h3 style="margin:0 0 4px;color:#00d2ff;font-size:18px;">ReAct Agent -- Watch It Think</h3>  <p style="margin:0 0 14px;font-size:13px;color:#888;">Query: What is India's population density?</p>  <div id="reactSteps" style="display:flex;flex-direction:column;gap:6px;"></div>  <button onclick="window._reactGo()" id="reactBtn" style="display:block;margin:14px auto 0;padding:10px 28px;background:linear-gradient(135deg,#00d2ff,#0891b2);color:#fff;border:none;border-radius:8px;cursor:pointer;font-size:14px;font-weight:600;">Run Agent</button></div><script>(function(){  var trace=[    {type:'think',text:'I need to find India\'s population first.',icon:'THINK',color:'#a78bfa'},    {type:'act',text:'search("India population 2024")',icon:'ACT',color:'#22c55e'},    {type:'observe',text:'India\'s population is approximately 1.44 billion.',icon:'OBS',color:'#fb923c'},    {type:'think',text:'Good. Now I need India\'s area in square km.',icon:'THINK',color:'#a78bfa'},    {type:'act',text:'search("India area square kilometers")',icon:'ACT',color:'#22c55e'},    {type:'observe',text:'India\'s area is 3,287,263 sq km.',icon:'OBS',color:'#fb923c'},    {type:'think',text:'I can calculate: 1,440,000,000 / 3,287,263',icon:'THINK',color:'#a78bfa'},    {type:'act',text:'calculate("1440000000 / 3287263")',icon:'ACT',color:'#22c55e'},    {type:'observe',text:'Result: 438.0',icon:'OBS',color:'#fb923c'},    {type:'answer',text:'India\'s population density is approximately 438 people per square kilometer.',icon:'DONE',color:'#00d2ff'}  ];  var shown=0,running=false;  function render(){    var html='<style>@keyframes fadeSlide{from{opacity:0;transform:translateY(-8px);}to{opacity:1;transform:translateY(0);}}</style>';    for(var i=0;i<shown;i++){      var s=trace[i];      var anim=i===shown-1?'animation:fadeSlide 0.4s ease;':'';      html+='<div style="display:flex;gap:10px;align-items:flex-start;padding:8px 12px;border-radius:8px;border-left:3px solid '+s.color+';background:'+s.color+'0a;'+anim+'">';      html+='<span style="font-size:11px;font-weight:bold;color:'+s.color+';min-width:40px;">'+s.icon+'</span>';      html+='<div style="font-size:13px;color:#ccc;">'+s.text+'</div></div>';    }    document.getElementById('reactSteps').innerHTML=html;  }  window._reactGo=function(){    if(running)return;    if(shown>=trace.length){shown=0;render();document.getElementById('reactBtn').textContent='Run Agent';return;}    running=true;document.getElementById('reactBtn').textContent='Thinking...';    function next(){shown++;render();if(shown<trace.length)setTimeout(next,900);else{running=false;document.getElementById('reactBtn').textContent='Reset and Replay';}}    next();  };  render();})();</script>""")

### ReAct vs Other Agent Architectures

| Architecture | How It Works | Pros | Cons |
|-------------|-------------|------|------|
| **ReAct** | Think → Act → Observe loop | Grounded, observable | Slow (many LLM calls) |
| **CoT** (Chain of Thought) | Think through entire problem | Fast (1 call) | Can hallucinate facts |
| **Plan-then-Execute** | Plan all steps → Execute all | Efficient | Rigid, can't adapt |
| **Reflexion** | ReAct + self-critique | Higher accuracy | Even more LLM calls |

### 🖼️ Agent Architectures — Visual Comparison

![Agent Architectures Compared](images/agent_architectures_comparison.png)

#### When to Use Each Architecture

| Scenario | Best Architecture | Why |
|----------|------------------|-----|
| Customer support chatbot | **ReAct** | Needs to look up different info based on each query |
| Math homework helper | **Chain-of-Thought** | Single reasoning chain, no external data needed |
| Report generation pipeline | **Plan-then-Execute** | Steps are predictable, can parallelize |
| Medical diagnosis assistant | **ReAct + Reflection** | Needs grounding AND self-checking |

In [ ]:
# Define tools for the research agent
import math

research_tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search for information on any topic. Returns relevant facts and data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform mathematical calculations. Supports arithmetic, powers, sqrt, trig, log.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression to evaluate"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function", 
        "function": {
            "name": "summarize",
            "description": "Summarize a piece of text into key bullet points.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "Text to summarize"},
                    "max_points": {"type": "integer", "description": "Maximum number of bullet points", "default": 5}
                },
                "required": ["text"]
            }
        }
    }
]

# Simulated search with more realistic data
def search(query):
    knowledge = {
        "population india": "India's population is approximately 1.44 billion (1,440,000,000) as of 2024, making it the most populous country.",
        "area india": "India's total area is 3,287,263 square kilometers (1,269,219 sq mi), the 7th largest country by area.",
        "gdp india": "India's GDP is approximately $3.7 trillion (nominal) in 2024, the 5th largest economy.",
        "python programming": "Python is a high-level programming language created by Guido van Rossum in 1991. It's the most popular language for AI/ML.",
        "machine learning": "Machine learning is a subset of AI where algorithms learn patterns from data. Key types: supervised, unsupervised, reinforcement learning.",
        "climate change": "Global temperatures have risen ~1.1°C since pre-industrial times. Major causes: fossil fuels, deforestation, industrial processes.",
        "spacex": "SpaceX is a private aerospace company founded by Elon Musk in 2002. Developed Falcon 9, Falcon Heavy, and Starship rockets.",
    }
    for key, value in knowledge.items():
        if key in query.lower():
            return json.dumps({"results": [{"text": value}], "query": query})
    return json.dumps({"results": [{"text": f"General information about: {query}"}], "query": query})

def calculate(expression):
    try:
        safe_dict = {"sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, "log": math.log, "pi": math.pi, "e": math.e, "abs": abs, "pow": pow}
        result = eval(expression.replace("^", "**"), {"__builtins__": {}}, safe_dict)
        return json.dumps({"expression": expression, "result": result})
    except Exception as e:
        return json.dumps({"error": str(e)})

def summarize(text, max_points=5):
    response = litellm.completion(
        model=DEFAULT_MODEL,
        messages=[{"role": "user", "content": f"Summarize in {max_points} bullet points:\n{text}"}],
        temperature=0, max_tokens=300,
    )
    return json.dumps({"summary": response.choices[0].message.content})

RESEARCH_TOOLS = {"search": search, "calculate": calculate, "summarize": summarize}

print("✅ Research tools ready")

In [ ]:
class ReActAgent:
    """A ReAct agent that reasons, acts, and observes."""
    
    def __init__(self, tools, tool_functions, model=None):
        self.tools = tools
        self.tool_functions = tool_functions
        self.model = model or DEFAULT_MODEL
        self.trace = []  # Full reasoning trace
    
    def run(self, query, max_rounds=8, verbose=True):
        """Execute the ReAct loop."""
        self.trace = []
        
        system = """You are a research assistant that solves problems step by step.

For each step:
1. THINK about what information you need
2. USE a tool to get that information
3. OBSERVE the result
4. Decide if you need more information or can give a final answer

Be thorough — use multiple tools when needed to build a complete answer."""
        
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": query}
        ]
        
        if verbose:
            print(f"\n🧠 ReAct Agent")
            print(f"❓ Query: {query}")
            print("─" * 60)
        
        for round_num in range(1, max_rounds + 1):
            response = litellm.completion(
                model=self.model, messages=messages,
                tools=self.tools, tool_choice="auto", temperature=0,
            )
            message = response.choices[0].message
            
            # No tool calls = final answer
            if not message.tool_calls:
                if verbose:
                    print(f"\n💬 FINAL ANSWER (Round {round_num}):")
                    print(f"   {message.content}")
                self.trace.append({"type": "answer", "round": round_num, "content": message.content})
                return {"answer": message.content, "trace": self.trace, "rounds": round_num}
            
            # Execute tools
            messages.append(message)
            for tc in message.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments)
                
                if verbose:
                    print(f"\n  🔧 Round {round_num} — {name}({json.dumps(args)[:50]})")
                
                try:
                    result = self.tool_functions[name](**args)
                except Exception as e:
                    result = json.dumps({"error": str(e)})
                
                if verbose:
                    print(f"     📋 {result[:100]}...")
                
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                self.trace.append({
                    "type": "tool_call", "round": round_num,
                    "tool": name, "args": args, "result": result
                })
        
        return {"answer": "Max rounds reached", "trace": self.trace, "rounds": max_rounds}

# Create and test the agent
research_agent = ReActAgent(research_tools, RESEARCH_TOOLS)

# Multi-step research query
result = research_agent.run(
    "What is India's population density (people per square km)? Show your calculation."
)

In [ ]:
# Test with a more complex multi-step query
print("\n" + "=" * 60)
result = research_agent.run(
    "Compare India's GDP per capita with its population density. What insights can we draw?"
)

# Print the full trace
print("\n📊 Agent Trace Summary:")
for step in result["trace"]:
    if step["type"] == "tool_call":
        print(f"  Round {step['round']}: {step['tool']}({list(step['args'].values())[0][:40]}...)")
    else:
        print(f"  Round {step['round']}: FINAL ANSWER")
print(f"  Total rounds: {result['rounds']}")

---
## 3. Agent Trace & Observability

In production, you MUST log every agent step for debugging and auditing.

In [ ]:
class ObservableAgent(ReActAgent):
    """Agent with full observability and metrics."""
    
    def run(self, query, max_rounds=8, verbose=True):
        start = time.time()
        result = super().run(query, max_rounds, verbose)
        
        # Add metrics
        result["latency_ms"] = int((time.time() - start) * 1000)
        result["tool_calls"] = sum(1 for t in result["trace"] if t["type"] == "tool_call")
        result["tools_used"] = list(set(t["tool"] for t in result["trace"] if t["type"] == "tool_call"))
        
        if verbose:
            print(f"\n📊 Metrics:")
            print(f"   Latency: {result['latency_ms']}ms")
            print(f"   Rounds: {result['rounds']}")
            print(f"   Tool calls: {result['tool_calls']}")
            print(f"   Tools used: {result['tools_used']}")
        
        return result

obs_agent = ObservableAgent(research_tools, RESEARCH_TOOLS)
result = obs_agent.run("How fast is SpaceX's Falcon 9 compared to the speed of sound?")

---
## 📝 Interview Quiz — ReAct Agents

### Q1: What is the ReAct pattern? How does it differ from Chain-of-Thought?

<details>
<summary>💡 Answer</summary>

**ReAct** = Reasoning + Acting in an interleaved loop:
- Thought → Action → Observation → Thought → ...
- The agent can use tools to gather real information
- Grounded in actual data

**Chain-of-Thought** = Pure reasoning without actions:
- Think through the entire problem in one pass
- No access to external tools/data
- Can hallucinate intermediate steps

**Key difference:** ReAct is *grounded* (uses real data from tools), CoT is *ungrounded* (relies on training data).
</details>

### Q2: When would you prefer Plan-then-Execute over ReAct?

<details>
<summary>💡 Answer</summary>

**Plan-then-Execute advantages:**
- More efficient (fewer LLM calls)
- Better for deterministic workflows
- Easier to parallelize steps

**Use when:**
- The steps are well-known and predictable
- Steps can run in parallel
- Latency matters more than flexibility

**Use ReAct when:**
- The next step depends on previous results
- The problem requires adaptive reasoning
- You need tool outputs to decide what to do next
</details>

### Q3: How do you prevent an agent from looping forever?

<details>
<summary>💡 Answer</summary>

1. **Max rounds** — Hard limit on iterations (e.g., 10)
2. **Token budget** — Stop after N total tokens consumed
3. **Time limit** — Timeout after N seconds
4. **Cycle detection** — Detect if the same tool is called with same args
5. **Progress check** — If no new information in last 2 rounds, force final answer
6. **Escalation** — After max retries, escalate to human
</details>

### Q4: How would you make a ReAct agent faster in production?

<details>
<summary>💡 Answer</summary>

1. **Parallel tool calls** — Some models support multiple simultaneous tool calls
2. **Smaller model for routing** — Use GPT-4o-mini for tool selection, larger model for final answer
3. **Caching** — Cache tool results for repeated queries
4. **Pre-computed context** — Pre-embed common queries
5. **Streaming** — Stream intermediate results to user while thinking
6. **Few-shot examples** — Help model learn to take fewer rounds
</details>

---
## 🏗️ BUILD: Research Assistant

In [ ]:
# Full research assistant with accumulated context
class ResearchAssistant:
    """A research assistant that builds a research report step by step."""
    
    def __init__(self, tools, tool_functions, model=None):
        self.agent = ObservableAgent(tools, tool_functions, model)
    
    def research(self, topic, sub_questions=None):
        """Conduct multi-query research on a topic."""
        if not sub_questions:
            # Generate sub-questions
            response = litellm.completion(
                model=DEFAULT_MODEL,
                messages=[{
                    "role": "user",
                    "content": f"Generate 3 specific research questions about: {topic}. Return as JSON array of strings."
                }],
                response_format={"type": "json_object"},
                temperature=0.3,
            )
            try:
                data = json.loads(response.choices[0].message.content)
                sub_questions = list(data.values())[0] if isinstance(list(data.values())[0], list) else [topic]
            except:
                sub_questions = [topic]
        
        print(f"📚 Research Topic: {topic}")
        print(f"📋 Sub-questions: {len(sub_questions)}")
        print("=" * 60)
        
        findings = []
        for i, q in enumerate(sub_questions, 1):
            print(f"\n── Question {i}/{len(sub_questions)} ──")
            result = self.agent.run(q, max_rounds=5, verbose=True)
            findings.append({"question": q, "answer": result["answer"], "rounds": result["rounds"]})
        
        return findings

assistant = ResearchAssistant(research_tools, RESEARCH_TOOLS)
findings = assistant.research("India's economy and demographics")

print("\n\n📋 Research Report")
print("=" * 60)
for f in findings:
    print(f"\n❓ {f['question']}")
    print(f"📝 {f['answer'][:200]}...")

---
## ✅ Notebook 08 Summary

| Concept | Key Takeaway |
|---------|-------------|
| ReAct | Thought → Action → Observation loop for grounded reasoning |
| vs CoT | ReAct uses real tools; CoT reasons from training data only |
| Multi-step | Chain tool calls to answer complex questions |
| Observability | Log every step: tool, args, result, latency |
| Safety | Max rounds, timeout, cycle detection |
| Research agent | Generate sub-questions, research each, synthesize |

### ➡️ Next: [Notebook 09 — Memory](./09_memory.ipynb)